# **No-arbitrage**

**MSc thesis Euan Bronsky**

**Notebook number: 4**

This notebook focuses exclusively on evaluating the no-arbitrage properties of the models. Specifically, it computes calendar and butterfly arbitrage violations and re-estimates the smoother model under different penalty values. This allows the trade-off between predictive accuracy and economic admissibility to be assessed separately.

--------------------------------------------------


- **Import dependencies**

In [1]:
%reset -f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sympy.abc import theta
from patsy import dmatrix
from statsmodels.tsa.api import VAR
import statsmodels.api as sm
import pickle
from patsy import build_design_matrices
from scipy.stats import norm
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
import statsmodels.formula.api as smf
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import random

- **Load the data**

In [2]:
# Load saved spline/state-space results
with open("/Users/euanbronsky/PyCharmMiscProject/4spline_results.pkl", "rb") as f:
    spline_results = pickle.load(f)

# Load saved VAR results
with open("/Users/euanbronsky/PyCharmMiscProject/2model1_results.pkl", "rb") as f:
    var_results = pickle.load(f)

# Load saved OU results
with open("/Users/euanbronsky/PyCharmMiscProject/3model_ou_results.pkl", "rb") as f:
    ou_results = pickle.load(f)

- **Extract Spline variables**

In [3]:
# Kalman/state-space coefficient paths
at_pred_spline = spline_results["at_pred"]

# Spline metadata
basis_cols = spline_results["basis_cols"]
m_bs_meta = spline_results["m_bs_meta"]
tau_bs_meta = spline_results["tau_bs_meta"]

# Spline datasets
x_train_spline = spline_results["x_train_spline"]
x_test_spline = spline_results["x_test_spline"]
x_full_spline = spline_results["x_full_spline"]
data_spline_full = spline_results["data_spline_full"]

# Dates
dates_spline = spline_results["dates"]

# **Preliminary functions**
----------------------------------
The preliminary functions set up the main building blocks needed for the neural network smoother.

- **Prepare group function**

In [4]:
# Prepare group function
def prepare_groups_spline_nn(x):

    # Sort the dates
    dates = np.sort(x["quote_date"].unique())

    # Create empty lists
    y_groups = []
    Z_groups = []

    # Loop over dates
    for date in dates:

        # Extract group
        group = x[x["quote_date"] == date]

        # Append results
        y_groups.append(group["log_iv"].to_numpy(dtype=float))
        Z_groups.append(group[basis_cols].to_numpy(dtype=float))

    # Return the results
    return dates, y_groups, Z_groups

- **Predict implied volatility function**

In [5]:
# Predict the IV function
def predict_iv(Z_groups, at_pred):

    # Create empty list
    log_iv_pred = []

    # Loop over the groups
    for t in range(len(Z_groups)):

        # Obtain design matrix
        Zt = Z_groups[t]

        # Obtain factors
        beta_t = at_pred[:, t]

        # Compute predictions
        log_iv_pred_t = Zt @ beta_t
        log_iv_pred.append(log_iv_pred_t)

    # Concatenate results and exponentiate
    log_iv_pred = np.concatenate(log_iv_pred)
    iv_pred = np.exp(log_iv_pred)

    # Return the results
    return log_iv_pred, iv_pred

# **Prepare neural network training**

----------------------------

The data are first prepared for the neural network training procedure. This step ensures that the model receives the correct inputs from the spline-based prior model and that the training and test samples are aligned consistently. In particular, the saved train/test split is loaded, the spline basis is reconstructed for the relevant observations, and the neural network class is defined.


- **Create full train and test samples**

In [6]:
# Training and testing dates
dates_train_spline = np.sort(x_train_spline["quote_date"].unique())
dates_test_spline = np.sort(x_test_spline["quote_date"].unique())

# Number of training dates
n_train_dates_spline = len(dates_train_spline)

# Training and testing sample
x_train_spline_full_nn = data_spline_full[data_spline_full["quote_date"].isin(dates_train_spline)].copy()
x_test_spline_full_nn = data_spline_full[data_spline_full["quote_date"].isin(dates_test_spline)].copy()

- **Enforce date order and sort**

In [7]:
# Ensure fixed quote date order for training sample
x_train_spline_full_nn["quote_date"] = pd.Categorical(x_train_spline_full_nn["quote_date"], categories=dates_train_spline, ordered=True)

# Ensure fixed quote date order for testing sample
x_test_spline_full_nn["quote_date"] = pd.Categorical(x_test_spline_full_nn["quote_date"], categories=dates_test_spline, ordered=True)

# Sort the values
x_train_spline_full_nn = x_train_spline_full_nn.sort_values("quote_date").copy()
x_test_spline_full_nn = x_test_spline_full_nn.sort_values("quote_date").copy()

# Convert back to string
x_train_spline_full_nn["quote_date"] = x_train_spline_full_nn["quote_date"].astype(str)
x_test_spline_full_nn["quote_date"] = x_test_spline_full_nn["quote_date"].astype(str)

- **Prepare groups, compute spline prior, and add date indices**

In [8]:
# Prepare training groups
dates_train_full_spline_nn, _, Z_groups_train_full_spline_nn = prepare_groups_spline_nn(x_train_spline_full_nn)

# Compute training predictions
_, iv_prior_train_full_spline_nn = predict_iv(Z_groups_train_full_spline_nn, at_pred_spline[:, :len(dates_train_full_spline_nn)])

# Prepare testing groups
dates_test_full_spline_nn, y_groups_test_full_spline_nn, Z_groups_test_full_spline_nn = prepare_groups_spline_nn(x_test_spline_full_nn)

# Compute testing predictions
log_iv_prior_test_full_spline_nn, iv_prior_test_full_spline_nn = predict_iv(Z_groups_test_full_spline_nn, at_pred_spline[:, n_train_dates_spline:n_train_dates_spline + len(dates_test_full_spline_nn)])

# Add date indices
x_train_spline_full_nn["date_idx"] = x_train_spline_full_nn.groupby("quote_date").ngroup()
x_test_spline_full_nn["date_idx"] = x_test_spline_full_nn.groupby("quote_date").ngroup()

- **Construct neural network**

In [9]:
# Define neural network class
class ResidualNN(nn.Module):

    # Initialize the neural network
    def __init__(self, n_features):

        # Initialize the parent PyTorch module
        super().__init__()

        # Define the hidden layers of the neural network
        self.hidden = nn.Sequential(

            # First hidden layer
            nn.Linear(n_features, 80),
            nn.Tanh(),

            # Second hidden layer
            nn.Linear(80, 80),
            nn.Tanh(),

            # Third hidden layer
            nn.Linear(80, 80),
            nn.Tanh())

        # Define output layer
        self.output = nn.Linear(80, 1)

        # Initialize the network weights
        self.initialize_weights()

    # Define the custom weight initialization function
    def initialize_weights(self):

        # Loop over all layers in the model
        for layer in self.modules():

            # Check whether current layer is a linear layer
            if isinstance(layer, nn.Linear):

                # Number of input and output units
                fan_in = layer.weight.shape[1]
                fan_out = layer.weight.shape[0]

                # Compute standard deviation for initialization
                std = (fan_in + fan_out) ** (-0.5)

                # Initialize layer weights at zero with standard deviation
                nn.init.normal_(layer.weight, mean=0.0, std=std)
                nn.init.normal_(layer.bias, mean=0.0, std=std)

        # Initialize output layers
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    # Define the forward pass of the neural network
    def forward(self, x):

        # Pass the inputs through the hidden layers
        x = self.hidden(x)

        # Compute the log multiplier
        log_omega_multiplier = self.output(x)

        # Exponentiate the log multiplier
        omega_multiplier = torch.exp(log_omega_multiplier)

        # Return the results
        return log_omega_multiplier, omega_multiplier

- **Recreate the spline basis, but using PyTorch**

The spline basis is reconstructed in PyTorch so that it remains fully differentiable. This is necessary because the no-arbitrage penalties require derivatives of the predicted total variance with respect to maturity and moneyness. By rebuilding the spline design matrix with Torch tensors, autograd can track these operations.

In [10]:
# Torch version of the B-spline basis function
def torch_bs_basis(x, bs_meta):

    # Flatten x into a one-dimensional tensor
    x = x.reshape(-1)

    # Create a Torch tensor containing all spline knots
    knots = torch.tensor(bs_meta["all_knots"], dtype=x.dtype, device=x.device)

    # Extract the spline degree
    degree = bs_meta["degree"]

    # Compute the number of B-spline basis function
    n_basis = len(knots) - degree - 1

    # Create an empty list to store the basis functions
    B = []

    # Construct the degree-zero b-spline basis functions
    for j in range(n_basis):

        # Indicator for whether x lies in the interval from knot j to j + 1
        Bj = ((x >= knots[j]) & (x < knots[j + 1])).to(x.dtype)

        # Store basis functions as a column
        B.append(Bj.reshape(-1, 1))

    # Combine all degree-zero basis functions into one matrix
    B = torch.cat(B, dim=1)

    # Recursively build higher-degree B-spline basis functions
    for d in range(1, degree + 1):

        # Create empty list
        B_new = []

        # Loop over each basis function
        for j in range(n_basis):

            # Initialize left and right part of recursion
            left = torch.zeros_like(x)
            right = torch.zeros_like(x)

            # Compute denominator of left and right terms
            left_denom = knots[j + d] - knots[j]
            right_denom = knots[j + d + 1] - knots[j + 1]

            # Add left recursion term when denominator is positive
            if left_denom > 0:
                left = ((x - knots[j]) / left_denom) * B[:, j]

            # Add right recursion term when denominator is positive
            if j + 1 < n_basis and right_denom > 0:
                right = ((knots[j + d + 1] - x) / right_denom) * B[:, j + 1]

            # Store updated basis function as a column
            B_new.append((left + right).reshape(-1, 1))

        # Combine all basis functions of degree d into one matrix
        B = torch.cat(B_new, dim=1)

    # Ensure that observations exactly at final knot belong to last basis function
    B[x == knots[-1], -1] = 1.0

    # Return result
    return B[:, 1:]

- **Spline design matrix construction**

In [11]:
# Torch spline design
def torch_remake_spline_design_from_tensors(M, T, basis_cols, m_bs_meta, tau_bs_meta):

    # Compute moneyness basis
    B_m = torch_bs_basis(M, m_bs_meta)

    # Compute maturity basis
    B_tau = torch_bs_basis(T, tau_bs_meta)

    # Create empty dictionary
    cols = {}

    # Add constant column
    cols["constant"] = torch.ones((M.numel(), 1), dtype=M.dtype, device=M.device)

    # Add moneyness basis columns
    for j in range(B_m.shape[1]):
        cols[f"m_basis_{j}"] = B_m[:, [j]]

    # Add maturity basis columns
    for h in range(B_tau.shape[1]):
        cols[f"tau_basis_{h}"] = B_tau[:, [h]]

    # Add interaction columns
    for j in range(B_m.shape[1]):

        # Loop over maturity basis columns for this moneyness basis column
        for h in range(B_tau.shape[1]):
            cols[f"m_basis_{j}_x_tau_basis_{h}"] = B_m[:, [j]] * B_tau[:, [h]]

    # Combine columns in the exact same order as original spline
    Z = torch.cat([cols[col] for col in basis_cols], dim=1)

    # Return the result
    return Z

# **2. Train the neural network**

--------------------------------

This function trains the neural network while incorporating the no-arbitrage penalties. In addition to minimizing the prediction error, the loss function penalizes violations of the calendar and butterfly arbitrage conditions, encouraging the fitted surface to be both accurate and economically admissible. Training is done multiple times for different values of the arbitrage penalties.

- **Training function**

In [12]:
# Neural network training function for various lambda values
def train_evaluate_smoother_nn(lambda_value, seed_value=8, batch_size_value=65536, n_synth_value=8192, max_epochs_value=400, lr_check_window_value=100, stop_check_window_value=200, required_improvement_value=0.003):

    # 1. Prepare training data

    # Copy full training sample and add spline prior predictions
    x_train_model = x_train_spline_full_nn.copy()
    x_train_model["iv_prior"] = iv_prior_train_full_spline_nn

    # Prior IV floor
    iv_prior_floor_model = 0.02
    x_train_model['iv_prior_safe'] = x_train_model['iv_prior'].clip(lower=iv_prior_floor_model)

    # Drop NaN's
    x_train_model = x_train_model.dropna(subset=["M", "T", "iv_fun", "iv_prior", "iv_prior_safe", "date_idx"]).copy()

    # Set seeds
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)

    # Convert implied volatility to Torch tensor
    y_tensor_model = torch.tensor(x_train_model[["iv_fun"]].to_numpy(dtype=float), dtype=torch.float32)

    # Convert moneyness to Torch tensor
    M_tensor_model = torch.tensor(x_train_model["M"].to_numpy(dtype=float), dtype=torch.float32)

    # Convert maturity to Torch tensor
    T_tensor_model = torch.tensor(x_train_model["T"].to_numpy(dtype=float), dtype=torch.float32)

    # Convert date indices to Torch tensor
    date_idx_tensor_model = torch.tensor(x_train_model["date_idx"].to_numpy(dtype=int), dtype=torch.long)

    # Build Torch dataset used by DataLoader
    dataset_model = TensorDataset(y_tensor_model, M_tensor_model, T_tensor_model, date_idx_tensor_model)

    # Build DataLoader
    loader_model = DataLoader(dataset_model, batch_size=batch_size_value, shuffle=True, generator=torch.Generator().manual_seed(seed_value))

    # Convert training-date spline coefficients to Torch tensor
    beta_train_tensor_model = torch.tensor(at_pred_spline[:, :len(dates_train_full_spline_nn)].T, dtype=torch.float32)

    # Convert testing-date spline coefficients to a Torch tensor
    beta_test_tensor_model = torch.tensor(at_pred_spline[:, n_train_dates_spline:n_train_dates_spline + len(dates_test_full_spline_nn)].T, dtype=torch.float32)

    # Compute training mean and standard deviation of moneyness
    M_mean_model = torch.tensor(x_train_model["M"].mean(), dtype=torch.float32)
    M_scale_model = torch.tensor(x_train_model["M"].std(ddof=0), dtype=torch.float32)

    # Compute training mean and standard deviation of maturity
    T_mean_model = torch.tensor(x_train_model["T"].mean(), dtype=torch.float32)
    T_scale_model = torch.tensor(x_train_model["T"].std(ddof=0), dtype=torch.float32)

    # Compute training mean and standard deviation of spline prior
    iv_prior_mean_model = torch.tensor(x_train_model["iv_prior_safe"].mean(), dtype=torch.float32)
    iv_prior_scale_model = torch.tensor(x_train_model["iv_prior_safe"].std(ddof=0), dtype=torch.float32)

    # 2. Create model and optimizer

    # Initialize neural network
    model_current = ResidualNN(n_features=3)

    # Set initial learning rate
    current_lr_model = 0.001

    # Define Adam optimizer
    optimizer_model = torch.optim.Adam(model_current.parameters(), lr=current_lr_model, weight_decay=1e-6)

    # Create list to store training loss history
    loss_history_model = []

    # 3. Synthetic grid

    # Compute transformed moneyness
    k_train_model = (x_train_model["M"].to_numpy(dtype=float) * np.sqrt(x_train_model["T"].to_numpy(dtype=float)))

    # Compute minimum and maximum moneyness and maturity
    k_min_model = float(np.nanmin(k_train_model))
    k_max_model = float(np.nanmax(k_train_model))
    T_max_model = float(np.nanmax(x_train_model["T"].to_numpy(dtype=float)))

    # Create an evenly spaced grid
    x_grid_model = np.linspace(-((-2.0 * k_min_model) ** (1.0 / 3.0)), (2.0 * k_max_model) ** (1.0 / 3.0), 100)

    # Convert grid back to the k scale
    k_grid_model = x_grid_model ** 3

    # Create an evenly spaced maturity grid
    T_grid_model = np.exp(np.linspace(np.log(1.0 / 365.0), np.log(T_max_model + 1.0), 100))

    # Create two-dimensional grid over values
    k_mesh_model, T_mesh_model = np.meshgrid(k_grid_model, T_grid_model,indexing="ij")

    # Convert moneyness grid to Torch tensor
    k_ic45_tensor_model = torch.tensor(k_mesh_model.reshape(-1), dtype=torch.float32)

    # Convert maturity grid to Torch tensor
    T_ic45_tensor_model = torch.tensor(T_mesh_model.reshape(-1), dtype=torch.float32)

    # Store number of grid points used for penalty
    n_ic45_model = len(k_ic45_tensor_model)

    # Create seeded Torch generator for reproducible grid
    grid_generator_model = torch.Generator().manual_seed(seed_value)

    # 4. Training loop

    # Loop over training epochs
    for epoch_model in range(max_epochs_value):

        # Initialize total loss for this epoch
        total_loss_model = 0.0

        # Loop over mini-batches
        for y_batch_model, M_batch_model, T_batch_model, date_idx_batch_model in loader_model:

            # Reset gradients before computing new batch gradient
            optimizer_model.zero_grad()

            # Gradients with respect to moneyness and maturity
            M_batch_model = M_batch_model.clone().detach().requires_grad_(True)
            T_batch_model = T_batch_model.clone().detach().requires_grad_(True)

            # Select spline coefficients
            beta_batch_model = beta_train_tensor_model[date_idx_batch_model]

            # Rebuild spline design matrix
            Z_batch_model = torch_remake_spline_design_from_tensors(M_batch_model, T_batch_model, basis_cols, m_bs_meta, tau_bs_meta)

            # Compute spline-prior IV
            iv_prior_model = torch.exp(torch.sum(Z_batch_model * beta_batch_model, dim=1, keepdim=True))
            iv_prior_model = torch.clamp(iv_prior_model, min=iv_prior_floor_model)

            # Build standardized neural network input features
            X_batch_model = torch.cat(
                [
                    (M_batch_model.reshape(-1, 1) - M_mean_model) / M_scale_model,
                    (T_batch_model.reshape(-1, 1) - T_mean_model) / T_scale_model,
                    (iv_prior_model - iv_prior_mean_model) / iv_prior_scale_model],dim=1)

            # Compute neural network multiplier
            _, omega_multiplier_model = model_current(X_batch_model)

            # Convert spline prior IV to total variance
            omega_prior_model = (T_batch_model.reshape(-1, 1) * iv_prior_model ** 2)

            # Apply neural network multiplier for predicted total variance
            omega_pred_model = (omega_prior_model * omega_multiplier_model)

            # Convert back to IV
            iv_pred_model = torch.sqrt(torch.clamp(omega_pred_model / torch.clamp(T_batch_model.reshape(-1, 1), min=1e-8), min=1e-12))

            # Compute RMSE
            rmse_model = torch.sqrt(torch.mean((y_batch_model - iv_pred_model) ** 2))

            # Compute MAPE
            mape_model = torch.mean(torch.abs(y_batch_model - iv_pred_model) / torch.clamp(y_batch_model, min=1e-8))

            # Initialize penalties
            calendar_penalty_model = torch.tensor(0.0, dtype=torch.float32)
            butterfly_penalty_model = torch.tensor(0.0, dtype=torch.float32)

            # No-arbitrage penalties on synthetic grid
            if lambda_value > 0:

                # Randomly sample grid locations
                grid_idx_model = torch.randint(low=0, high=n_ic45_model, size=(n_synth_value,), generator=grid_generator_model)

                # Randomly sample dates
                date_pos_model = torch.randint(low=0, high=len(date_idx_batch_model), size=(n_synth_value,), generator=grid_generator_model)

                # Select transformed moneyness values
                k_synth_model = (k_ic45_tensor_model[grid_idx_model].clone().detach().requires_grad_(True))

                # Select maturity values
                T_synth_model = (T_ic45_tensor_model[grid_idx_model].clone().detach().requires_grad_(True))

                # Convert transformed moneyness back to moneyness
                M_synth_model = k_synth_model / torch.sqrt(T_synth_model)

                # Select spline coefficients
                beta_synth_model = beta_train_tensor_model[date_idx_batch_model[date_pos_model]]

                # Rebuild spline design matrix
                Z_synth_model = torch_remake_spline_design_from_tensors(M_synth_model, T_synth_model, basis_cols, m_bs_meta, tau_bs_meta)

                # Compute spline prior IV on grid
                iv_prior_synth_model = torch.exp(torch.sum(Z_synth_model * beta_synth_model, dim=1, keepdim=True))
                iv_prior_synth_model = torch.clamp(iv_prior_synth_model, min=iv_prior_floor_model)

                # Build standardized neural network input features
                X_synth_model = torch.cat(
                    [
                        (M_synth_model.reshape(-1, 1) - M_mean_model) / M_scale_model,
                        (T_synth_model.reshape(-1, 1) - T_mean_model) / T_scale_model,
                        (iv_prior_synth_model - iv_prior_mean_model) / iv_prior_scale_model], dim=1)

                # Compute neural network multiplier
                _, omega_multiplier_synth_model = model_current(X_synth_model)

                # Convert spline prior IV into total variance
                omega_prior_synth_model = (T_synth_model.reshape(-1, 1) * iv_prior_synth_model ** 2)

                # Apply neural network multiplier
                omega_synth_model = (omega_multiplier_synth_model * omega_prior_synth_model)

                # Compute derivative of total variance with respect to maturity
                l_cal_model = torch.autograd.grad(omega_synth_model.sum(), T_synth_model, create_graph=True)[0].reshape(-1, 1)

                # Compute first derivative of total variance with respect to moneyness
                omega_k_model = torch.autograd.grad(omega_synth_model.sum(),k_synth_model, create_graph=True)[0].reshape(-1, 1)

                # Second derivative
                omega_kk_model = torch.autograd.grad(omega_k_model.sum(), k_synth_model, create_graph=True)[0].reshape(-1, 1)

                # Reshape moneyness into column vector
                k_col_model = k_synth_model.reshape(-1, 1)
                omega_safe_model = torch.clamp(omega_synth_model, min=1e-8)

                # Compute butterfly arbitrage expression
                l_but_model = ((1 - (k_col_model * omega_k_model) / (2 * omega_safe_model)) ** 2 - ((omega_k_model ** 2) / 4) * ((1 / omega_safe_model) + 0.25) + 0.5 * omega_kk_model)

                # Compute penalties
                calendar_penalty_model = torch.mean(torch.relu(-l_cal_model))
                butterfly_penalty_model = torch.mean(torch.relu(-l_but_model))

            # Use only prediction losses when no arbitrage penalty is applied
            if lambda_value == 0:
                loss_model = rmse_model + mape_model

            # Add penalties else
            else:
                loss_model = (rmse_model + mape_model + lambda_value * calendar_penalty_model + lambda_value * butterfly_penalty_model)

            # Backpropagate the loss and update parameters
            loss_model.backward()
            optimizer_model.step()

            # Add current loss to total
            total_loss_model += loss_model.item()

        # Compute average loss and append
        avg_loss_model = total_loss_model / len(loader_model)
        loss_history_model.append(avg_loss_model)

        # Print progress
        if epoch_model % 50 == 0:
            print("lambda", lambda_value, "epoch", epoch_model, "loss", avg_loss_model, "lr:", current_lr_model)

        # Check whether learning rate should be reduced
        if epoch_model > 0 and epoch_model % lr_check_window_value == 0:

            # Get loss from previous learning-rate check window
            old_loss_model = loss_history_model[epoch_model - lr_check_window_value]

            # Compute relative old loss improvement
            improvement_model = (old_loss_model - avg_loss_model) / old_loss_model

            # Reduce learning rate if improvement is too small
            if improvement_model < required_improvement_value:

                # Halve the current learning rate
                current_lr_model = current_lr_model / 2

                # Update optimizer with new learning rate
                for param_group_model in optimizer_model.param_groups:
                    param_group_model["lr"] = current_lr_model

                # Print reduction message
                print("lambda", lambda_value, "reduced learning rate to", current_lr_model, "at epoch", epoch_model)

        # Check whether early stopping should be applied
        if epoch_model > 0 and epoch_model % stop_check_window_value == 0:

            # Get loss from previous stopping check window
            old_loss_model = loss_history_model[epoch_model - stop_check_window_value]

            # Compute relative loss improvement
            improvement_model = (old_loss_model - avg_loss_model) / old_loss_model

            # Stop if improvement is too small
            if improvement_model < required_improvement_value:
                print("lambda", lambda_value, "stopping: loss did not improve enough")
                break

    # 5. Test-set prediction

    # Copy the full testing sample and add spline prior
    x_test_model = x_test_spline_full_nn.copy()
    x_test_model["iv_prior"] = iv_prior_test_full_spline_nn
    x_test_model["iv_prior_safe"] = x_test_model["iv_prior"].clip(lower=iv_prior_floor_model)

    # Drop NaN's
    x_test_model = x_test_model.dropna(
        subset=["M", "T", "iv_fun", "iv_prior", "iv_prior_safe", "date_idx"]
    ).copy()

    # Convert testing moneyness to Torch tensor
    M_test_tensor_model = torch.tensor(x_test_model["M"].to_numpy(dtype=float), dtype=torch.float32)

    # Convert testing maturity to a Torch tensor
    T_test_tensor_model = torch.tensor(x_test_model["T"].to_numpy(dtype=float),dtype=torch.float32)

    # Convert testing dates to Torch tensor
    date_idx_test_tensor_model = torch.tensor(x_test_model["date_idx"].to_numpy(dtype=int), dtype=torch.long)

    # Switch the neural network to evaluation mode
    model_current.eval()

    # Disable gradient tracking
    with torch.no_grad():

        # Select testing spline coefficients
        beta_test_batch_model = beta_test_tensor_model[date_idx_test_tensor_model]

        # Rebuild spline design matrix for testing observations
        Z_test_model = torch_remake_spline_design_from_tensors(M_test_tensor_model, T_test_tensor_model, basis_cols, m_bs_meta, tau_bs_meta)

        # Compute spline prior IV
        iv_prior_test_model = torch.exp(torch.sum(Z_test_model * beta_test_batch_model, dim=1, keepdim=True))
        iv_prior_test_model = torch.clamp(iv_prior_test_model, min=iv_prior_floor_model)

        # Build standardized neural network input features
        X_test_model = torch.cat(
            [
                (M_test_tensor_model.reshape(-1, 1) - M_mean_model) / M_scale_model,
                (T_test_tensor_model.reshape(-1, 1) - T_mean_model) / T_scale_model,
                (iv_prior_test_model - iv_prior_mean_model) / iv_prior_scale_model], dim=1)

        # Compute multiplier
        _, omega_multiplier_test_model = model_current(X_test_model)

        # Convert to total variance
        omega_prior_test_model = (T_test_tensor_model.reshape(-1, 1) * iv_prior_test_model ** 2)

        # Apply multiplier
        omega_test_model = (omega_prior_test_model * omega_multiplier_test_model)

        # Convert back to implied volatility
        iv_nn_test_model = torch.sqrt(torch.clamp(omega_test_model / torch.clamp(T_test_tensor_model.reshape(-1, 1), min=1e-8), min=1e-12))

    # Store neural network implied predictions in testing sample
    x_test_model["iv_nn"] = iv_nn_test_model.cpu().numpy().flatten()
    x_test_model["abs_error_nn"] = np.abs(x_test_model["iv_fun"] - x_test_model["iv_nn"])

    # Compute mean absolute error on testing sample
    mae_model = x_test_model["abs_error_nn"].mean()

    # 6. Test-set arbitrage violations

    # Extract testing moneyness, maturity, and transformed moneyness values
    M_obs_model = x_test_model["M"].to_numpy(dtype=float)
    T_obs_model = x_test_model["T"].to_numpy(dtype=float)
    k_obs_model = M_obs_model * np.sqrt(T_obs_model)

    # Convert to Torch tensor
    k_arb_model = torch.tensor(k_obs_model, dtype=torch.float32).requires_grad_(True)
    T_arb_model = torch.tensor(T_obs_model, dtype=torch.float32).requires_grad_(True)

    # Convert back to moneyness
    M_arb_model = k_arb_model / torch.sqrt(T_arb_model)

    # Convert data to Torch tensor
    date_idx_arb_model = torch.tensor(x_test_model["date_idx"].to_numpy(dtype=int), dtype=torch.long)

    # Select testing spline coefficients
    beta_arb_model = beta_test_tensor_model[date_idx_arb_model]

    # Rebuild spline prior implied volatility
    Z_arb_model = torch_remake_spline_design_from_tensors(M_arb_model, T_arb_model, basis_cols, m_bs_meta, tau_bs_meta)

    # Compute spline prior IV for arbitrage check
    iv_prior_arb_model = torch.exp(torch.sum(Z_arb_model * beta_arb_model, dim=1, keepdim=True))
    iv_prior_arb_model = torch.clamp(iv_prior_arb_model, min=iv_prior_floor_model)

    # Build standardized network inputs
    X_arb_model = torch.cat(
        [
            (M_arb_model.reshape(-1, 1) - M_mean_model) / M_scale_model,
            (T_arb_model.reshape(-1, 1) - T_mean_model) / T_scale_model,
            (iv_prior_arb_model - iv_prior_mean_model) / iv_prior_scale_model], dim=1)

    # Compute multiplier
    _, omega_multiplier_arb_model = model_current(X_arb_model)

    # Convert to total variance
    omega_prior_arb_model = (T_arb_model.reshape(-1, 1) * iv_prior_arb_model ** 2)

    # Apply multiplier
    omega_arb_model = (omega_prior_arb_model * omega_multiplier_arb_model)

    # Compute derivative
    l_cal_test_model = torch.autograd.grad(omega_arb_model.sum(), T_arb_model, create_graph=True)[0].reshape(-1, 1)

    # Compute derivative
    omega_k_test_model = torch.autograd.grad(omega_arb_model.sum(), k_arb_model, create_graph=True)[0].reshape(-1, 1)

    # Compute derivative
    omega_kk_test_model = torch.autograd.grad(omega_k_test_model.sum(), k_arb_model, create_graph=False)[0].reshape(-1, 1)

    # Clamp the total variance
    omega_safe_test_model = torch.clamp(omega_arb_model, min=1e-8)
    k_col_test_model = k_arb_model.reshape(-1, 1)

    # Compute butterfly arbitrage expression
    l_but_test_model = ((1 - (k_col_test_model * omega_k_test_model) / (2 * omega_safe_test_model)) ** 2 - ((omega_k_test_model ** 2) / 4) * ((1 / omega_safe_test_model) + 0.25) + 0.5 * omega_kk_test_model)

    # Set tolerance
    tol_model = 1e-8

    # Count number of calendar violations
    n_calendar_violations_model = (l_cal_test_model < -tol_model).sum().item()
    calendar_violation_rate_model = (l_cal_test_model < -tol_model).float().mean().item()

    # Count number of butterfly violations
    n_butterfly_violations_model = (l_but_test_model < -tol_model).sum().item()
    butterfly_violation_rate_model = (l_but_test_model < -tol_model).float().mean().item()

    # Store results in DataFrame
    x_test_model["k_log_forward"] = k_obs_model
    x_test_model["l_cal"] = l_cal_test_model.detach().cpu().numpy().flatten()
    x_test_model["l_but"] = l_but_test_model.detach().cpu().numpy().flatten()

    # Print values
    print("lambda", lambda_value, "MAE:", mae_model)
    print("lambda", lambda_value, "calendar violations:", n_calendar_violations_model)
    print("lambda", lambda_value, "butterfly violations:", n_butterfly_violations_model)

    # Return the results
    return {
        "x_train": x_train_model,
        "x_test": x_test_model,
        "mae": mae_model,
        "n_calendar_violations": n_calendar_violations_model,
        "calendar_violation_rate": calendar_violation_rate_model,
        "n_butterfly_violations": n_butterfly_violations_model,
        "butterfly_violation_rate": butterfly_violation_rate_model}

- **Print the results**

In [13]:
# Create empty dictionary
penalty_results = {}

# Loop over lambda values
for lambda_value in [0.0, 10.0, 20.0, 50.0, 100.0]:

    # Apply the function
    penalty_results[f"lambda_{int(lambda_value)}"] = train_evaluate_smoother_nn(
        lambda_value=lambda_value)

lambda 0.0 epoch 0 loss 0.17924696717943464 lr: 0.001
lambda 0.0 epoch 50 loss 0.05789021253585815 lr: 0.001
lambda 0.0 epoch 100 loss 0.05762319532888276 lr: 0.001
lambda 0.0 epoch 150 loss 0.057329474495989935 lr: 0.001
lambda 0.0 epoch 200 loss 0.057199726892369136 lr: 0.001
lambda 0.0 epoch 250 loss 0.05723170063325337 lr: 0.001
lambda 0.0 epoch 300 loss 0.05722382834979466 lr: 0.001
lambda 0.0 reduced learning rate to 0.0005 at epoch 300
lambda 0.0 epoch 350 loss 0.056878543006522314 lr: 0.0005
lambda 0.0 MAE: 0.013184083716086776
lambda 0.0 calendar violations: 44789
lambda 0.0 butterfly violations: 122759
lambda 10.0 epoch 0 loss 0.18789457253047398 lr: 0.001
lambda 10.0 epoch 50 loss 0.06714706122875214 lr: 0.001
lambda 10.0 epoch 100 loss 0.06456371332917894 lr: 0.001
lambda 10.0 epoch 150 loss 0.06201145840542657 lr: 0.001
lambda 10.0 epoch 200 loss 0.07017025319593294 lr: 0.001
lambda 10.0 reduced learning rate to 0.0005 at epoch 200
lambda 10.0 epoch 250 loss 0.060229940499

- **Save the results**

In [14]:
with open("/Users/euanbronsky/PyCharmMiscProject/penalty_results.pkl", "wb") as f:
    pickle.dump(penalty_results, f)

- **Display error metrics**

In [15]:
# Create an empty list
rows = []

# Loop over each lambda result
for key, result in penalty_results.items():

    # Extract the test-set DataFrame
    x = result["x_test"]

    # Compute daily DOTM put MAE
    dotm_put_mae = (
        x[x["moneyness_group"] == "DOTM_put"]
        .groupby("quote_date")["abs_error_nn"]
        .mean()
        .mean())

    # Compute aggregate MAE
    aggregate_mae = (
        x.groupby("quote_date")["abs_error_nn"]
        .mean()
        .mean())

    # Append results
    rows.append([key, dotm_put_mae, aggregate_mae])

# Convert results into DataFrame and print
mae_summary_by_lambda = pd.DataFrame(rows, columns=["lambda", "DOTM_put_MAE", "Aggregate_MAE"])
mae_summary_by_lambda

,lambda,DOTM_put_MAE,Aggregate_MAE
0,lambda_0,0.024630,0.014046
1,lambda_10,0.024676,0.014244
2,lambda_20,0.024926,0.014365
3,lambda_50,0.025213,0.014884
4,lambda_100,0.026657,0.015689


# **Compute number of benchmark arbitrage violations**

-------------------

To assess how the neural smoother model compares with the benchmark models in terms of economic admissibility, the number of arbitrage violations is computed for each benchmark specification. These violations provide a direct measure of how often the predicted implied volatility surfaces imply option prices that fail the imposed no-arbitrage condition.

- **No-arbitrage counting function**

In [16]:
# No-arbitrage counting function
def check_arbitrage_violations(x_test_full, at_pred, n_train_dates, name):

    # Sort the data
    df = x_test_full.sort_values(["quote_date", "moneyness_group", "maturity_group"]).copy()

    # Extract moneyness and maturity
    M = df["M"].to_numpy(dtype=float)
    T = df["T"].to_numpy(dtype=float)

    # Compute square-root maturity and convert to log-forward moneyness
    sqrt_T = np.sqrt(T)
    k = M * sqrt_T

    # Extract unique test dates
    dates_test = np.sort(df["quote_date"].unique())

    # Select predicted factor values for test period
    beta_test = at_pred[:, n_train_dates:n_train_dates + len(dates_test)].T

    # Store predicted factor values
    beta_df = pd.DataFrame(beta_test, index=dates_test, columns=["b0", "b1", "b2", "b3", "b4"])

    # Add predicted factor values to option-level data
    df = df.join(beta_df, on="quote_date")

    # Extract factor values as NumPy arrays
    b0 = df["b0"].to_numpy(dtype=float)
    b1 = df["b1"].to_numpy(dtype=float)
    b2 = df["b2"].to_numpy(dtype=float)
    b3 = df["b3"].to_numpy(dtype=float)
    b4 = df["b4"].to_numpy(dtype=float)

    # Compute fitted log implied volatility
    f = b0 + b1 * M + b2 * M ** 2 + b3 * T + b4 * M * T

    # Convert fitted log implied volatility into IV
    iv = np.exp(f)

    # Compute total variance
    omega = T * iv ** 2
    omega_safe = np.maximum(omega, 1e-12)

    # Compute derivatives with respect to M
    f_M = b1 + 2 * b2 * M + b4 * T
    f_MM = 2 * b2
    f_T_at_M = b3 + b4 * M

    # Convert derivatives into k-space
    f_k = f_M / sqrt_T
    f_kk = f_MM / T

    # Calendar derivative must hold k fixed
    dM_dT_at_k = -M / (2 * T)

    # Compute derivative with respect to T, holding k fixed
    f_T_at_k = f_T_at_M + f_M * dM_dT_at_k

    # Compute calendar condition
    l_cal = np.exp(2 * f) * (1 + 2 * T * f_T_at_k)

    # Butterfly derivatives wrt k
    omega_k = 2 * omega * f_k
    omega_kk = omega * (2 * f_kk + 4 * f_k ** 2)

    # Compute butterfly condition
    l_but = ((1 - (k * omega_k) / (2 * omega_safe)) ** 2 - ((omega_k ** 2) / 4) * ((1 / omega_safe) + 0.25) + 0.5 * omega_kk)

    # Store the results
    df["iv"] = iv
    df["omega"] = omega
    df["l_cal"] = l_cal
    df["l_but"] = l_but

    # Compute number of violations
    df["calendar_violation"] = df["l_cal"] < 0
    df["butterfly_violation"] = df["l_but"] < 0

    # Print the results
    print(f"{name} calendar violations:", df["calendar_violation"].sum())
    print(f"{name} calendar violation rate:", df["calendar_violation"].mean())
    print(f"{name} calendar violation rate (%):", 100 * df["calendar_violation"].mean())
    print(f"{name} butterfly violations:", df["butterfly_violation"].sum())
    print(f"{name} butterfly violation rate:", df["butterfly_violation"].mean())
    print(f"{name} butterfly violation rate (%):", 100 * df["butterfly_violation"].mean())

    # Return the results
    return df

- **Prepare group function**

In [17]:
# Group preparing function
def prepare_groups(x):

    # Sort the data
    x = x.sort_values(['quote_date', 'moneyness_group', 'maturity_group']).reset_index(drop=True)

    # Extract dates and create empty lists
    dates = np.sort(x['quote_date'].unique())
    y_groups = []
    Z_groups = []

    # Loop over dates
    for date in dates:

        # Extract groups
        group = x[x['quote_date'] == date]

        # Extract relevant variables and append to lists
        y_groups.append(group['log_iv'].to_numpy(dtype=float))
        Z_groups.append(group[['constant', 'M', 'moneyness_squared', 'T', 'interaction']].to_numpy(dtype=float))

    # return the results
    return dates, y_groups, Z_groups

- **Prepare VAR data**

In [19]:
# Import data
data = pd.read_parquet("/Users/euanbronsky/Downloads/data_final_final.parquet")

# Retrieve VAR objects
at_pred_var = var_results["at_pred"]
x_train_var = var_results["x_train"]
x_test_var = var_results["x_test"]

# Prepare VAR bucketed train and test dates
dates_train_var, _, _ = prepare_groups(x_train_var)
n_train_dates_var = len(dates_train_var)
dates_test_var, _, _ = prepare_groups(x_test_var)

# Recreate the full-contract VAR test sample from the original full dataset
x_test_full_var = data[data["quote_date"].isin(dates_test_var)].copy()
x_test_full_var = x_test_full_var.sort_values(
    ["quote_date", "moneyness_group", "maturity_group"]
).copy()

- **Prepare OU data**

In [26]:
# Retrieve OU objects
at_pred_ou = ou_results["at_pred"]

# Prepare OU bucketed train and test dates
dates_train_ou, _, _ = prepare_groups(x_train_var)
n_train_dates_ou = len(dates_train_ou)
dates_test_ou, _, _ = prepare_groups(x_test_var)

# Recreate the full-contract OU test sample from the original full dataset
x_test_full_ou = data[data["quote_date"].isin(dates_test_ou)].copy()
x_test_full_ou = x_test_full_ou.sort_values(
    ["quote_date", "moneyness_group", "maturity_group"]
).copy()

- **Apply the functions**

In [27]:
# VAR model
df_var = check_arbitrage_violations(x_test_full_var, at_pred_var, n_train_dates_var, "VAR")

# OU model
df_ou = check_arbitrage_violations(x_test_full_ou, at_pred_ou, n_train_dates_ou, "OU")

VAR calendar violations: 54423
VAR calendar violation rate: 0.01672535995803217
VAR calendar violation rate (%): 1.6725359958032169
VAR butterfly violations: 34556
VAR butterfly violation rate: 0.010619803000748943
VAR butterfly violation rate (%): 1.0619803000748944
OU calendar violations: 15985
OU calendar violation rate: 0.004912534754224211
OU calendar violation rate (%): 0.49125347542242115
OU butterfly violations: 13249
OU butterfly violation rate: 0.004071703031511828
OU butterfly violation rate (%): 0.4071703031511828


- **Spline model**

In [25]:
# Extract observed spline-model moneyness values
M_obs_spline = x_test_spline_full_nn["M"].to_numpy(dtype=float)
T_obs_spline = x_test_spline_full_nn["T"].to_numpy(dtype=float)

# Recover true forward log-moneyness k
k_obs_spline = M_obs_spline * np.sqrt(T_obs_spline)

# Convert to Torch tensor
k_test_tensor_spline_arb = torch.tensor(k_obs_spline,dtype=torch.float32).requires_grad_(True)

# Convert to Torch tensor
T_test_tensor_spline_arb = torch.tensor(T_obs_spline,dtype=torch.float32).requires_grad_(True)

# Convert true k back to model input M
M_test_tensor_spline_arb = k_test_tensor_spline_arb / torch.sqrt(T_test_tensor_spline_arb)

# Convert to Torch tensor
date_idx_test_tensor_spline_arb = torch.tensor(x_test_spline_full_nn["date_idx"].to_numpy(dtype=int), dtype=torch.long)

# Convert to Torch tensor
beta_test_tensor_spline_arb = torch.tensor(at_pred_spline[:, n_train_dates_spline:n_train_dates_spline + len(dates_test_full_spline_nn)].T, dtype=torch.float32)

# Select spline coefficients
beta_test_batch_spline_arb = beta_test_tensor_spline_arb[date_idx_test_tensor_spline_arb]

# Rebuild spline design matrix
Z_test_spline_arb = torch_remake_spline_design_from_tensors(M_test_tensor_spline_arb, T_test_tensor_spline_arb, basis_cols, m_bs_meta, tau_bs_meta)

# Compute log implied volatility from spline model and exponentiate
log_iv_spline_arb = torch.sum(Z_test_spline_arb * beta_test_batch_spline_arb,dim=1, keepdim=True)
iv_spline_arb = torch.exp(log_iv_spline_arb)

# Convert IV to total variance
omega_spline_arb = (T_test_tensor_spline_arb.reshape(-1, 1) * iv_spline_arb ** 2)

# Compute calendar derivative
l_cal_test_spline = torch.autograd.grad(omega_spline_arb.sum(), T_test_tensor_spline_arb,create_graph=True)[0].reshape(-1, 1)

# Compute first derivative with respect to moneyness
omega_k_test_spline = torch.autograd.grad(omega_spline_arb.sum(), k_test_tensor_spline_arb, create_graph=True)[0].reshape(-1, 1)

# Compute second derivative with respect to moneyness
omega_kk_test_spline = torch.autograd.grad(omega_k_test_spline.sum(),k_test_tensor_spline_arb, create_graph=False)[0].reshape(-1, 1)

# Reshape to column vector and clip
k_test_spline = k_test_tensor_spline_arb.reshape(-1, 1)
omega_safe_test_spline = torch.clamp(omega_spline_arb, min=1e-8)

# Compute butterfly arbitrage expression
l_but_test_spline = ((1 - (k_test_spline * omega_k_test_spline) / (2 * omega_safe_test_spline)) ** 2 - ((omega_k_test_spline ** 2) / 4) * ((1 / omega_safe_test_spline) + 0.25) + 0.5 * omega_kk_test_spline)

# Set tolerance
tol_spline = 1e-8

# Count calendar arbitrage violations and violation rate
n_calendar_violations_spline = (l_cal_test_spline < -tol_spline).sum().item()
calendar_violation_rate_spline = (l_cal_test_spline < -tol_spline).float().mean().item()

# Count butterfly arbitrage violations and violation rate
n_butterfly_violations_spline = (l_but_test_spline < -tol_spline).sum().item()
butterfly_violation_rate_spline = (l_but_test_spline < -tol_spline).float().mean().item()

# Store diagnostics
x_test_spline_full_nn["k_log_forward"] = k_obs_spline
x_test_spline_full_nn["l_cal_spline"] = l_cal_test_spline.detach().cpu().numpy().flatten()
x_test_spline_full_nn["l_but_spline"] = l_but_test_spline.detach().cpu().numpy().flatten()

# Print the results
print("Spline calendar violations:", n_calendar_violations_spline)
print("Spline calendar violation rate:", calendar_violation_rate_spline)
print("Spline calendar violation rate (%):", 100 * calendar_violation_rate_spline)
print("Spline butterfly violations:", n_butterfly_violations_spline)
print("Spline butterfly violation rate:", butterfly_violation_rate_spline)
print("Spline butterfly violation rate (%):", 100 * butterfly_violation_rate_spline)

Spline calendar violations: 11309
Spline calendar violation rate: 0.003974979743361473
Spline calendar violation rate (%): 0.3974979743361473
Spline butterfly violations: 76640
Spline butterfly violation rate: 0.026938052847981453
Spline butterfly violation rate (%): 2.6938052847981453


# **Diebold-Mariano test**

----------------------------
To assess whether the observed differences in relative forecast errors are statistically significant, Diebold-Mariano tests are conducted using the non-penalized smoother as the reference model. This comparison evaluates whether the arbitrage-penalized specifications produce forecast errors that differ significantly from the unconstrained neural smoother.

- **Align losses**

In [28]:
# Align losses
def align_losses(losses):

    # Initialize common index
    common = None

    # Loop over all loss series
    for loss_series in losses.values():

        # Use the first model's dates as the starting set of dates
        if common is None:
            common = loss_series.index

        # Keep only dates that are common across all models
        else:
            common = common.intersection(loss_series.index)

    # Sort the common dates
    common = common.sort_values()

    # Store aligned losses in a DataFrame
    loss_df = pd.DataFrame({
        name: loss_series.loc[common].to_numpy()
        for name, loss_series in losses.items()
    }, index=common)

    # Return aligned losses
    return loss_df

- **DM test function**

In [29]:
# Diebold-Mariano test function
def dm_test_loss(loss_model, loss_benchmark, lag=5):

    # Compute the loss differential
    loss_diff = np.asarray(loss_model) - np.asarray(loss_benchmark)

    # Length of the series, mean differential, and demean
    T = len(loss_diff)
    d_bar = np.mean(loss_diff)
    u = loss_diff - d_bar

    # Average squared deviation
    lrv = np.mean(u * u)

    # HAC standard errors
    for j in range(1, lag + 1):

        # Compute gamma and autocovariance corrections
        gamma = np.mean(u[j:] * u[:-j])
        lrv += 2 * (1 - j / (lag + 1)) * gamma

    # Compute standard error, test statistic, and p-value
    se = np.sqrt(lrv / T)
    dm_stat = d_bar / se
    p_value = 2 * (1 - norm.cdf(np.abs(dm_stat)))

    # Return the results
    return dm_stat, p_value


- **DM table function**

In [30]:
# DM table function relative to the non-penalized smoother
def dm_table_penalties(loss_df, benchmark="lambda_0", lag=5):

    # Create empty list
    out = []

    # Loop over columns
    for name in loss_df.columns:

        # Skip benchmark model itself
        if name == benchmark:
            continue

        # Run DM test for current model
        dm_stat, p_value = dm_test_loss(loss_df[name].to_numpy(), loss_df[benchmark].to_numpy(),lag=lag)

        # Store results in list
        out.append([name, loss_df[name].mean(), dm_stat, p_value])

    # Return the results
    return pd.DataFrame(
        out,
        columns=[
            "model",
            "mean MAE",
            "DM statistic",
            "p-value"])

- **Daily loss series for DM test**

In [31]:
# Daily MAE loss for a penalty specification
def daily_penalty_mae(result, group=None):

    # Copy data
    x = result["x_test"].copy()

    # Filter for specific region
    if group is not None:
        x = x[x["moneyness_group"] == group]

    # Return the results
    return x.groupby("quote_date")["abs_error_nn"].mean()

- **Sort penalty specifications and align losses**

In [32]:
# Sort penalty specifications by lambda value
penalty_keys = sorted(
    penalty_results.keys(),
    key=lambda x: int(x.replace("lambda_", "")))

# Construct aligned daily MAE losses
loss_penalty_agg_mae = align_losses({
    key: daily_penalty_mae(penalty_results[key])
    for key in penalty_keys})

# Region-specific daily MAE losses
loss_penalty_dotm_mae = align_losses({
    key: daily_penalty_mae(penalty_results[key], group="DOTM_put")
    for key in penalty_keys})

- **Run the tests and display results**

In [33]:
# Run DM tests relative to the non-penalized smoother
dm_penalty_agg_mae = dm_table_penalties(loss_penalty_agg_mae, benchmark="lambda_0", lag=5)
dm_penalty_dotm_mae = dm_table_penalties(loss_penalty_dotm_mae, benchmark="lambda_0", lag=5)

# Print results
print(dm_penalty_agg_mae)
print(dm_penalty_dotm_mae)

        model  mean MAE  DM statistic       p-value
0   lambda_10  0.014244      2.286619  2.221809e-02
1   lambda_20  0.014365      3.175573  1.495407e-03
2   lambda_50  0.014884      4.326874  1.512404e-05
3  lambda_100  0.015689      5.263884  1.410435e-07
        model  mean MAE  DM statistic       p-value
0   lambda_10  0.024676      0.420449  6.741578e-01
1   lambda_20  0.024926      2.017664  4.362627e-02
2   lambda_50  0.025213      2.662647  7.752877e-03
3  lambda_100  0.026657      5.971084  2.356824e-09
